# Project 10 — AI Agent

## Overview
We build an **AI agent** that can use tools to accomplish multi-step tasks. The agent uses Claude's tool-calling API to reason about what to do, call tools, observe results, and iterate until the task is complete.

**Agent loop:**
```
Goal → Think (Claude) → Tool Call → Observe Result → Think → ... → Final Answer
```

**Tools implemented:**
- `calculator` — evaluate math expressions
- `get_current_time` — current date/time
- `search_wikipedia` — look up information
- `run_python_code` — execute Python snippets
- `read_file` — read local files

## 1. Theory — AI Agents

### What is an Agent?
An agent is a system that **perceives** its environment, **reasons** about it, and **acts** to achieve a goal. In LLM-based agents, the LLM serves as the reasoning engine.

### ReAct Framework
ReAct (Reasoning + Acting) interleaves reasoning traces and actions:
```
Thought: I need to find what year Python was created.
Action: search_wikipedia("Python programming language")
Observation: Python was created in 1991 by Guido van Rossum.
Thought: Now I can answer the question.
Answer: Python was created in 1991.
```

### Tool Calling
Modern LLMs support **structured function calling**. You define tools as JSON schemas:
```json
{"name": "calculator",
 "description": "Evaluate a math expression",
 "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}}}}
```
The model returns structured tool calls which your code executes, then feeds results back.

### Memory Types
| Type | Description | Scope |
|---|---|---|
| **Scratchpad** | Current reasoning trace | Current task |
| **Episodic** | Past task memories | Session |
| **Semantic** | Factual knowledge | Long-term |
| **Procedural** | Tool/action skills | Persistent |

### Agent Failure Modes
- **Infinite loops**: agent calls the same tool repeatedly
- **Hallucinated tools**: agent tries to call non-existent tools
- **Context overflow**: too many tool results fill the context window
- **Goal drift**: agent pursues sub-goals instead of the main objective

**Mitigations**: max iterations, tool allow-lists, context summarisation, explicit goal tracking.

In [ ]:
# Required: pip install anthropic
import ast
import json
import math
import os
import subprocess
import sys
import textwrap
import time
import urllib.request
from datetime import datetime
from typing import Any, Dict, List, Optional

import anthropic

# Initialise Anthropic client — uses ANTHROPIC_API_KEY env var
client = anthropic.Anthropic()
MODEL  = 'claude-haiku-4-5-20251001'  # fast and cheap for agent loops

print(f'Anthropic SDK version: {anthropic.__version__}')
print(f'Using model: {MODEL}')

## 2. Tool Implementations

In [ ]:
def calculator(expression: str) -> str:
    """
    Safely evaluate a mathematical expression.
    Supports basic arithmetic, math functions (sqrt, log, etc.).
    """
    # Safe math globals — prevents access to builtins
    safe_globals = {
        '__builtins__': {},
        'abs': abs, 'round': round, 'min': min, 'max': max,
        'sum': sum, 'pow': pow,
        **{k: getattr(math, k) for k in dir(math) if not k.startswith('_')}
    }
    try:
        result = eval(expression, safe_globals, {})
        return str(result)
    except Exception as e:
        return f'Error: {e}'


def get_current_time() -> str:
    """Return the current date and time."""
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')


def search_wikipedia(query: str) -> str:
    """
    Search Wikipedia and return the summary of the most relevant article.
    Uses the Wikipedia REST API — no API key required.
    """
    try:
        # Wikipedia REST API for search
        import urllib.parse
        search_url = f'https://en.wikipedia.org/api/rest_v1/page/summary/{urllib.parse.quote(query)}'
        req  = urllib.request.Request(search_url, headers={'User-Agent': 'AI-Agent/1.0'})
        with urllib.request.urlopen(req, timeout=10) as response:
            data = json.loads(response.read().decode())
        return data.get('extract', 'No information found.')[:500]
    except Exception as e:
        # Fallback: return hardcoded facts for common queries
        facts = {
            'python': 'Python is a high-level programming language created by Guido van Rossum, first released in 1991.',
            'machine learning': 'Machine learning is a subset of AI that enables systems to learn from data, first formalised in the 1950s.',
            'transformer': 'The Transformer architecture was introduced in 2017 in the paper "Attention Is All You Need" by Vaswani et al.',
        }
        for key, fact in facts.items():
            if key.lower() in query.lower():
                return fact
        return f'Search failed: {e}. Please try a different query.'


def run_python_code(code: str) -> str:
    """
    Execute Python code in a subprocess and return stdout/stderr.
    Timeout: 10 seconds.
    """
    try:
        result = subprocess.run(
            [sys.executable, '-c', code],
            capture_output=True,
            text=True,
            timeout=10
        )
        output = result.stdout or result.stderr or '(no output)'
        return output.strip()[:1000]  # limit output length
    except subprocess.TimeoutExpired:
        return 'Error: Code execution timed out (10s limit).'
    except Exception as e:
        return f'Error: {e}'


def read_file(filepath: str) -> str:
    """
    Read a local file and return its contents.
    Only allows reading .txt and .py files for safety.
    """
    allowed_extensions = {'.txt', '.py', '.md', '.json', '.csv'}
    if not any(filepath.endswith(ext) for ext in allowed_extensions):
        return f'Error: Only {allowed_extensions} files are allowed.'
    try:
        with open(filepath, 'r') as f:
            content = f.read()
        return content[:2000]  # limit to 2000 chars
    except FileNotFoundError:
        return f'Error: File not found: {filepath}'
    except Exception as e:
        return f'Error: {e}'


# Test the tools
print('calculator("15 * 0.15")  =>', calculator('15 * 0.15'))
print('calculator("sqrt(144)")  =>', calculator('sqrt(144)'))
print('get_current_time()       =>', get_current_time())
print('run_python_code(...)     =>', run_python_code('print(sum(range(10)))'))

## 3. Tool Schemas for Claude

In [ ]:
TOOL_SCHEMAS = [
    {
        'name': 'calculator',
        'description': 'Evaluate a mathematical expression. Supports arithmetic, math functions (sqrt, log, sin, etc.).',
        'input_schema': {
            'type': 'object',
            'properties': {
                'expression': {
                    'type': 'string',
                    'description': 'Mathematical expression to evaluate, e.g. "15 * 0.15" or "sqrt(144) + log(100)"'
                }
            },
            'required': ['expression']
        }
    },
    {
        'name': 'get_current_time',
        'description': 'Get the current date and time.',
        'input_schema': {
            'type': 'object',
            'properties': {},
            'required': []
        }
    },
    {
        'name': 'search_wikipedia',
        'description': 'Search Wikipedia for information about a topic. Returns a summary paragraph.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {
                    'type': 'string',
                    'description': 'The topic to search for'
                }
            },
            'required': ['query']
        }
    },
    {
        'name': 'run_python_code',
        'description': 'Execute Python code and return stdout output. Useful for calculations, data processing, testing algorithms.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'code': {
                    'type': 'string',
                    'description': 'Python code to execute'
                }
            },
            'required': ['code']
        }
    },
]

# Map tool names to functions
TOOL_FUNCTIONS = {
    'calculator':      calculator,
    'get_current_time': get_current_time,
    'search_wikipedia': search_wikipedia,
    'run_python_code':  run_python_code,
}

print(f'{len(TOOL_SCHEMAS)} tools registered.')

## 4. Agent Implementation

In [ ]:
class Agent:
    """
    ReAct-style AI agent powered by Claude.

    The agent loop:
    1. Send messages + tools to Claude
    2. Claude either responds with text (final answer) or requests tool calls
    3. Execute tool calls, append results as tool_result messages
    4. Repeat until final text response or max_iterations reached
    """

    def __init__(self, tools: list, model: str, max_iterations: int = 10):
        self.tools          = tools
        self.model          = model
        self.max_iterations = max_iterations
        self.client         = anthropic.Anthropic()

    def run(self, task: str, verbose: bool = True) -> str:
        """Run the agent on a task and return the final answer."""
        messages   = [{'role': 'user', 'content': task}]
        iterations = 0

        if verbose:
            print(f'\nTask: {task}')
            print('=' * 60)

        while iterations < self.max_iterations:
            iterations += 1

            # Call Claude with available tools
            response = self.client.messages.create(
                model=self.model,
                max_tokens=1024,
                tools=self.tools,
                messages=messages
            )

            # Append assistant response to conversation
            messages.append({'role': 'assistant', 'content': response.content})

            # Check stop reason
            if response.stop_reason == 'end_turn':
                # Extract final text answer
                final = next(
                    (b.text for b in response.content if hasattr(b, 'text')),
                    'No answer provided.'
                )
                if verbose:
                    print(f'\nFinal Answer: {final}')
                return final

            if response.stop_reason == 'tool_use':
                # Execute all requested tool calls
                tool_results = []
                for block in response.content:
                    if block.type == 'tool_use':
                        tool_name = block.name
                        tool_args = block.input

                        if verbose:
                            print(f'[Iter {iterations}] Tool: {tool_name}({json.dumps(tool_args)})')

                        # Execute the tool
                        if tool_name in TOOL_FUNCTIONS:
                            func = TOOL_FUNCTIONS[tool_name]
                            if tool_args:
                                result = func(**tool_args)
                            else:
                                result = func()
                        else:
                            result = f'Error: Unknown tool {tool_name}'

                        if verbose:
                            print(f'         → {str(result)[:100]}')

                        tool_results.append({
                            'type':       'tool_result',
                            'tool_use_id': block.id,
                            'content':    str(result)
                        })

                # Add tool results as a user message
                messages.append({'role': 'user', 'content': tool_results})

        return f'Reached max iterations ({self.max_iterations}). Last state: {messages[-1]}'


# Create agent instance
agent = Agent(tools=TOOL_SCHEMAS, model=MODEL, max_iterations=10)
print('Agent created.')

## 5. Demo Tasks

In [ ]:
# Task 1: Multi-step math
result = agent.run(
    'What is 15% of 847? Then compute the square root of that result. Show both values.',
    verbose=True
)

In [ ]:
# Task 2: Time-aware reasoning
result = agent.run(
    'What is the current year? Then tell me how many years ago the Python programming language was first released.',
    verbose=True
)

In [ ]:
# Task 3: Code execution
result = agent.run(
    'Write and run Python code to compute the first 10 Fibonacci numbers, then tell me the 10th one.',
    verbose=True
)

## Summary

| Component | Details |
|---|---|
| LLM | Claude Haiku (fast, cheap for agent loops) |
| Tools | calculator, time, wikipedia, python executor |
| Loop | Max 10 iterations, stop on `end_turn` |
| Safety | Sandboxed subprocess, allowed file extensions |

**Key insight**: The tool-calling protocol is the crucial interface. The LLM decides *which* tools to call and *when* — your code just executes them and reports back. This separation makes agents modular: you can add any tool without changing the agent loop.